In [1]:
! pip install optuna


In [2]:
import optuna
from sklearn import datasets
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from sklearn.svm import SVC
from sklearn.model_selection import cross_val_score
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC

In [3]:
df = breast_cancer = datasets.load_breast_cancer()

In [4]:
df = pd.DataFrame(breast_cancer.data, columns=breast_cancer.feature_names)
df['target'] = breast_cancer.target


In [5]:
df.head()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,0.14710,0.2419,0.07871,...,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890,0
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,0.07017,0.1812,0.05667,...,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902,0
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,0.12790,0.2069,0.05999,...,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758,0
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,0.10520,0.2597,0.09744,...,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300,0
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,0.10430,0.1809,0.05883,...,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678,0


In [6]:
df.describe()

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension,target
count,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,...,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000,569.000000
mean,14.127292,19.289649,91.969033,654.889104,0.096360,0.104341,0.088799,0.048919,0.181162,0.062798,...,25.677223,107.261213,880.583128,0.132369,0.254265,0.272188,0.114606,0.290076,0.083946,0.627417
std,3.524049,4.301036,24.298981,351.914129,0.014064,0.052813,0.079720,0.038803,0.027414,0.007060,...,6.146258,33.602542,569.356993,0.022832,0.157336,0.208624,0.065732,0.061867,0.018061,0.483918
min,6.981000,9.710000,43.790000,143.500000,0.052630,0.019380,0.000000,0.000000,0.106000,0.049960,...,12.020000,50.410000,185.200000,0.071170,0.027290,0.000000,0.000000,0.156500,0.055040,0.000000
25%,11.700000,16.170000,75.170000,420.300000,0.086370,0.064920,0.029560,0.020310,0.161900,0.057700,...,21.080000,84.110000,515.300000,0.116600,0.147200,0.114500,0.064930,0.250400,0.071460,0.000000
50%,13.370000,18.840000,86.240000,551.100000,0.095870,0.092630,0.061540,0.033500,0.179200,0.061540,...,25.410000,97.660000,686.500000,0.131300,0.211900,0.226700,0.099930,0.282200,0.080040,1.000000
75%,15.780000,21.800000,104.100000,782.700000,0.105300,0.130400,0.130700,0.074000,0.195700,0.066120,...,29.720000,125.400000,1084.000000,0.146000,0.339100,0.382900,0.161400,0.317900,0.092080,1.000000
max,28.110000,39.280000,188.500000,2501.000000,0.163400,0.345400,0.426800,0.201200,0.304000,0.097440,...,49.540000,251.200000,4254.000000,0.222600,1.058000,1.252000,0.291000,0.663800,0.207500,1.000000


In [7]:
df.isnull().sum()

,0
mean radius,0
mean texture,0
mean perimeter,0
mean area,0
mean smoothness,0
mean compactness,0
mean concavity,0
mean concave points,0
mean symmetry,0
mean fractal dimension,0


In [8]:
X = df.drop('target', axis=1)
y = df['target']

In [9]:
X

,mean radius,mean texture,mean perimeter,mean area,mean smoothness,mean compactness,mean concavity,mean concave points,mean symmetry,mean fractal dimension,...,worst radius,worst texture,worst perimeter,worst area,worst smoothness,worst compactness,worst concavity,worst concave points,worst symmetry,worst fractal dimension
0,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.30010,0.14710,0.2419,0.07871,...,25.380,17.33,184.60,2019.0,0.16220,0.66560,0.7119,0.2654,0.4601,0.11890
1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.08690,0.07017,0.1812,0.05667,...,24.990,23.41,158.80,1956.0,0.12380,0.18660,0.2416,0.1860,0.2750,0.08902
2,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.19740,0.12790,0.2069,0.05999,...,23.570,25.53,152.50,1709.0,0.14440,0.42450,0.4504,0.2430,0.3613,0.08758
3,11.42,20.38,77.58,386.1,0.14250,0.28390,0.24140,0.10520,0.2597,0.09744,...,14.910,26.50,98.87,567.7,0.20980,0.86630,0.6869,0.2575,0.6638,0.17300
4,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.19800,0.10430,0.1809,0.05883,...,22.540,16.67,152.20,1575.0,0.13740,0.20500,0.4000,0.1625,0.2364,0.07678
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
564,21.56,22.39,142.00,1479.0,0.11100,0.11590,0.24390,0.13890,0.1726,0.05623,...,25.450,26.40,166.10,2027.0,0.14100,0.21130,0.4107,0.2216,0.2060,0.07115
565,20.13,28.25,131.20,1261.0,0.09780,0.10340,0.14400,0.09791,0.1752,0.05533,...,23.690,38.25,155.00,1731.0,0.11660,0.19220,0.3215,0.1628,0.2572,0.06637
566,16.60,28.08,108.30,858.1,0.08455,0.10230,0.09251,0.05302,0.1590,0.05648,...,18.980,34.12,126.70,1124.0,0.11390,0.30940,0.3403,0.1418,0.2218,0.07820
567,20.60,29.33,140.10,1265.0,0.11780,0.27700,0.35140,0.15200,0.2397,0.07016,...,25.740,39.42,184.60,1821.0,0.16500,0.86810,0.9387,0.2650,0.4087,0.12400


In [10]:
X_train ,X_test ,y_train ,y_test = train_test_split(X,y,test_size=0.3,random_state=42)

In [11]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((398, 30), (171, 30), (398,), (171,))

In [12]:
def objective(trial):
 # Define the hyperparameters Value
  n_estimators = trial.suggest_int('n_estimators',50,200)
  max_depth = int(trial.suggest_float('max_depth', 3,20))


  # Create the RandomForestClassifier Hyperparmeter Value
  model = RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth=max_depth,
      random_state=42
  )
  scores = cross_val_score(model,X_train,y_train,cv=3,scoring='accuracy').mean()

  return scores


In [13]:
# Create a study Oject and optimize function
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.TPESampler())
study.optimize(objective,n_trials=50)

[I 2026-07-03 18:58:06,187] A new study created in memory with name: no-name-0fcf51fa-87aa-44c3-8693-020897f4e3ed
[I 2026-07-03 18:58:07,461] Trial 0 finished with value: 0.949741778689147 and parameters: {'n_estimators': 151, 'max_depth': 4.200664064209353}. Best is trial 0 with value: 0.949741778689147.
[I 2026-07-03 18:58:08,526] Trial 1 finished with value: 0.9522480443533073 and parameters: {'n_estimators': 118, 'max_depth': 9.87311218477936}. Best is trial 1 with value: 0.9522480443533073.
[I 2026-07-03 18:58:10,901] Trial 2 finished with value: 0.9597858282068809 and parameters: {'n_estimators': 195, 'max_depth': 9.483257523494022}. Best is trial 2 with value: 0.9597858282068809.
[I 2026-07-03 18:58:13,116] Trial 3 finished with value: 0.9572605756816284 and parameters: {'n_estimators': 145, 'max_depth': 12.988930299393939}. Best is trial 2 with value: 0.9597858282068809.
[I 2026-07-03 18:58:17,365] Trial 4 finished with value: 0.9547732968785599 and parameters: {'n_estimators':

In [14]:
# Print results from the study that used sampler=optuna.samplers.TPESampler()

# Print the best score (accuracy) achieved across all 50 trials
print(f'Best trial: {study.best_trial.value}')

# Print the hyperparameter values (n_estimators, max_depth) that produced that best score
print(f'Best params: {study.best_trial.params}')

Best trial: 0.9623300675932255
Best params: {'n_estimators': 67, 'max_depth': 19.449698919108464}


In [15]:
# Copy the best hyperparameters found by Optuna (TPESampler search)
# .copy() avoids accidentally modifying study.best_trial.params directly
best_params = study.best_trial.params.copy()

# Convert max_depth back to int
# (it was suggested as a float via trial.suggest_float, so RandomForest needs it as int)
best_params['max_depth'] = int(best_params['max_depth'])

# Create a new RandomForestClassifier using the best hyperparameters found
# **best_params unpacks the dict into n_estimators=..., max_depth=...
best_model = RandomForestClassifier(**best_params, random_state=42)

# Train the final model on the full training set using the best hyperparameters
best_model.fit(X_train, y_train)

# Use the trained model to make predictions on the unseen test set
y_pred = best_model.predict(X_test)

# Compare predicted labels vs actual labels to compute accuracy
accuracy = accuracy_score(y_test, y_pred)

# Print the final test accuracy of the tuned model
print(f'Accuracy: {accuracy}')

Accuracy: 0.9707602339181286


In [16]:
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.RandomSampler())
study.optimize(objective,n_trials=50)

[I 2026-07-03 18:58:52,635] A new study created in memory with name: no-name-fd9cc2cd-cd64-4a97-b681-d83f03cc7eeb
[I 2026-07-03 18:58:53,571] Trial 0 finished with value: 0.9572795625427205 and parameters: {'n_estimators': 152, 'max_depth': 17.111214933854093}. Best is trial 0 with value: 0.9572795625427205.
[I 2026-07-03 18:58:54,492] Trial 1 finished with value: 0.9572795625427205 and parameters: {'n_estimators': 148, 'max_depth': 17.04965432718779}. Best is trial 0 with value: 0.9572795625427205.
[I 2026-07-03 18:58:55,280] Trial 2 finished with value: 0.9572795625427205 and parameters: {'n_estimators': 78, 'max_depth': 6.125650486376997}. Best is trial 0 with value: 0.9572795625427205.
[I 2026-07-03 18:58:56,038] Trial 3 finished with value: 0.9522480443533073 and parameters: {'n_estimators': 74, 'max_depth': 13.103950024999548}. Best is trial 0 with value: 0.9572795625427205.
[I 2026-07-03 18:58:56,571] Trial 4 finished with value: 0.9597858282068809 and parameters: {'n_estimators

In [17]:
print(f'Best trial: {study.best_trial.value}')
print(f'Best params: {study.best_trial.params}')

Best trial: 0.9623300675932255
Best params: {'n_estimators': 67, 'max_depth': 9.041604811718102}


In [18]:
best_params = study.best_trial.params.copy()
best_params['max_depth'] = int(best_params['max_depth'])
best_model = RandomForestClassifier(**best_params, random_state=42)

best_model.fit(X_train,y_train)

y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)

print(f'Accuracy: {accuracy}')

Accuracy: 0.9707602339181286


In [19]:
# Define the grid of hyperparameter values to search over
# GridSampler will try EVERY combination of these values (4 x 4 = 16 total trials)
search_space = {
    'n_estimators': [50, 100, 150, 200],  # number of trees/boosting rounds to try
    'max_depth': [3, 10, 15, 20],         # max depth of each tree to try
}

In [20]:
study = optuna.create_study(direction='maximize',sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective,n_trials=50)

[I 2026-07-03 18:59:35,898] A new study created in memory with name: no-name-cb9a0d7f-1b3c-4a9d-823c-cb44197907c0
[I 2026-07-03 18:59:36,827] Trial 0 finished with value: 0.9447292473608263 and parameters: {'n_estimators': 100, 'max_depth': 3}. Best is trial 0 with value: 0.9447292473608263.
[I 2026-07-03 18:59:37,740] Trial 1 finished with value: 0.9572795625427205 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 1 with value: 0.9572795625427205.
[I 2026-07-03 18:59:38,079] Trial 2 finished with value: 0.9622920938710413 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 2 with value: 0.9622920938710413.
[I 2026-07-03 18:59:38,690] Trial 3 finished with value: 0.9572985494038125 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 2 with value: 0.9622920938710413.
[I 2026-07-03 18:59:39,326] Trial 4 finished with value: 0.9572985494038125 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 2 with value: 0.962292

In [21]:
print(f'Best trial: {study.best_trial.value}')
print(f'Best params: {study.best_trial.params}')

Best trial: 0.9622920938710413
Best params: {'n_estimators': 50, 'max_depth': 15.0}


In [22]:
best_params = study.best_trial.params.copy()
best_params['max_depth'] = int(best_params['max_depth'])
best_model = RandomForestClassifier(**best_params, random_state=42)

best_model.fit(X_train,y_train)

y_pred = best_model.predict(X_test)

accuracy = accuracy_score(y_test,y_pred)

print(f'Accuracy: {accuracy}')

Accuracy: 0.9707602339181286


In [23]:
from optuna.visualization import plot_optimization_history, plot_param_importances, plot_parallel_coordinate, plot_slice,plot_contour

In [24]:
plot_optimization_history(study).show()

In [25]:
plot_parallel_coordinate(study).show()

In [26]:
plot_slice(study).show()

In [27]:
plot_contour(study).show()

In [28]:
plot_param_importances(study).show()

In [29]:
def objective(trial):
  classifier_name = trial.suggest_categorical('classifier', ['SVC', 'RandomForest','GradientBoosting'])

  if classifier_name == 'SVC':
    svc_c = trial.suggest_float('svc_c', 0.1, 100, log=True)
    kernal = trial.suggest_categorical('kernal', ['linear', 'poly', 'rbf', 'sigmoid'])
    gama = trial.suggest_categorical('gama', ['scale', 'auto'])
    model = SVC(C=svc_c, kernel=kernal, gamma=gama, random_state=42)

  elif classifier_name == 'RandomForest':
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    max_depth = int(trial.suggest_float('max_depth', 3, 20))
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    boostrap = trial.suggest_categorical('boostrap', [True, False])

    model = RandomForestClassifier(
      n_estimators=n_estimators,
      max_depth=max_depth,
      min_samples_split=min_samples_split,
      min_samples_leaf=min_samples_leaf,
      bootstrap=boostrap,
      random_state=42
    )

  elif classifier_name == 'GradientBoosting':
    n_estimators = trial.suggest_int('n_estimators', 50, 300)
    learning_rate = trial.suggest_float('learning_rate', 0.001, 1.0, log=True)
    max_depth = int(trial.suggest_float('max_depth', 3, 20))
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    min_samples_leaf = trial.suggest_int('min_samples_leaf', 1, 20)
    subsample = trial.suggest_float('subsample', 0.1, 1.0)

    model = GradientBoostingClassifier(
      n_estimators=n_estimators,
      learning_rate=learning_rate,
      max_depth=max_depth,
      min_samples_split=min_samples_split,
      min_samples_leaf=min_samples_leaf,
      subsample=subsample,
      random_state=42
    )

  scores = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

  return scores

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-07-03 18:59:52,095] A new study created in memory with name: no-name-cf8dc39e-fe5c-4a53-90f9-e5378f4ae8b8
[I 2026-07-03 18:59:52,130] Trial 0 finished with value: 0.8743259664312295 and parameters: {'classifier': 'SVC', 'svc_c': 0.15097780667771366, 'kernal': 'rbf', 'gama': 'scale'}. Best is trial 0 with value: 0.8743259664312295.
[I 2026-07-03 18:59:52,169] Trial 1 finished with value: 0.7304055593529277 and parameters: {'classifier': 'SVC', 'svc_c': 74.43574031071705, 'kernal': 'sigmoid', 'gama': 'scale'}. Best is trial 0 with value: 0.8743259664312295.
[I 2026-07-03 18:59:56,233] Trial 2 finished with value: 0.8720285562390826 and parameters: {'classifier': 'GradientBoosting', 'n_estimators': 283, 'learning_rate': 0.7372089823185175, 'max_depth': 9.095427291597634, 'min_samples_split': 12, 'min_samples_leaf': 3, 'subsample': 0.5999489455179838}. Best is trial 0 with value: 0.8743259664312295.
[I 2026-07-03 18:59:56,283] Trial 3 finished with value: 0.6256360598465861 and par

In [ ]:
print(f'Best trial: {best_trial.value}')
print(f'Best params: {best_trial.params}')

In [ ]:
study.trials_dataframe()